# 04 — Securing the MCP Server

**What you'll learn**
- Why an unauthenticated HTTP MCP server is dangerous on day one.
- How to add bearer-token auth as middleware around the server.
- What a failed auth flow should look like to a client.
- A glimpse of scopes (read vs write tokens) — full treatment in notebook 05.

## Threat model in one paragraph

Your MCP server can `create_contact`, `assign_osc`, and `create_followup_task`. Those write to your CRM. If a stranger can reach the URL, they can spam your CRM with junk leads, reassign every contact to themselves, or simply rate-limit you off your own platform. Authentication is the **first** thing you add after picking HTTP — not the tenth.

## Architecture with auth

```text
+-----------+   Authorization: Bearer <token>    +----------------+
| Host /    |  --------------------------------> | Auth middleware|
| client    |  <-- 401 if missing/invalid ------ +--------+-------+
+-----------+                                              |
                                                           v
                                                  +----------------+
                                                  | MCP server     |
                                                  +----------------+
```

The middleware lives **outside** the tool functions. The tools stay clean; the wrapper decides who is allowed in.

In [ ]:
from typing import Callable, Any

class MiniMCPServer:
    """Tiny MCP-style server used for teaching.

    Real MCP defines a JSON-RPC protocol on top of a transport (stdio or HTTP).
    This class keeps only the three ideas you actually need to internalize:
      1. tools are registered with a name + description + schema
      2. a client can list them
      3. a client can call one by name with arguments
    """

    def __init__(self, name: str) -> None:
        self.name = name
        self._tools: dict[str, Callable[..., Any]] = {}
        self._descriptions: dict[str, str] = {}

    def tool(self, description: str = ""):
        """Decorator that registers a function as an MCP tool."""
        def decorator(func: Callable[..., Any]) -> Callable[..., Any]:
            self._tools[func.__name__] = func
            self._descriptions[func.__name__] = description or (func.__doc__ or "").strip()
            return func
        return decorator

    def list_tools(self) -> list[dict]:
        """Discovery: what can I do?"""
        return [{"name": n, "description": self._descriptions[n]} for n in self._tools]

    def call_tool(self, name: str, arguments: dict) -> Any:
        """Execution: do the thing."""
        if name not in self._tools:
            raise ValueError(f"Unknown tool: {name}")
        return self._tools[name](**arguments)

In [ ]:
# Fake in-memory CRM. In production this would be HubSpot, Salesforce, etc.
CONTACTS: dict = {}
TASKS: list = []
OSC_TEAM = [
    {"id": "osc_101", "name": "Ava OSC",  "last_assigned": 0},
    {"id": "osc_102", "name": "Ben OSC",  "last_assigned": 0},
    {"id": "osc_103", "name": "Cara OSC", "last_assigned": 0},
]
_assignment_counter = 0
print("Fake CRM ready. Contacts:", len(CONTACTS), "OSCs:", len(OSC_TEAM))

In [ ]:
server = MiniMCPServer("sales-tools-auth")

@server.tool(description="Create a new contact in the CRM")
def create_contact(name: str, email: str) -> dict:
    contact_id = f"contact_{len(CONTACTS) + 1}"
    contact = {"id": contact_id, "name": name, "email": email, "owner_id": None}
    CONTACTS[contact_id] = contact
    return contact

@server.tool(description="Assign an OSC to a contact using round-robin")
def assign_osc(contact_id: str) -> dict:
    global _assignment_counter
    chosen = min(OSC_TEAM, key=lambda osc: osc["last_assigned"])
    _assignment_counter += 1
    chosen["last_assigned"] = _assignment_counter
    CONTACTS[contact_id]["owner_id"] = chosen["id"]
    return chosen

@server.tool(description="Create a follow-up task for a contact")
def create_followup_task(contact_id: str, note: str) -> dict:
    task = {"id": f"task_{len(TASKS) + 1}", "contact_id": contact_id,
            "note": note, "status": "open"}
    TASKS.append(task)
    return task

server.list_tools()

## The auth wrapper

A tiny class that sits in front of the server. Every call must carry headers; if the token isn't in our allowlist, we reject the call before the tool ever runs.

In [ ]:
class UnauthorizedError(Exception):
    """Raised when a request lacks a valid token. In real MCP this maps to a 401."""

VALID_TOKENS = {
    # token -> human label (so logs are useful)
    "tok_alice_readwrite": "alice@acme.com",
    "tok_bot_pipeline":    "lead-pipeline-bot",
}


class AuthenticatedMCPServer:
    """Wraps a MiniMCPServer. Requires a valid bearer token on every call."""

    def __init__(self, inner: MiniMCPServer):
        self._inner = inner

    def _authorize(self, headers: dict | None) -> str:
        headers = headers or {}
        raw = headers.get("Authorization") or headers.get("authorization")
        if not raw or not raw.startswith("Bearer "):
            raise UnauthorizedError("missing or malformed Authorization header")
        token = raw.removeprefix("Bearer ").strip()
        if token not in VALID_TOKENS:
            raise UnauthorizedError("invalid token")
        return VALID_TOKENS[token]

    def list_tools(self, headers: dict | None = None) -> list[dict]:
        user = self._authorize(headers)
        print(f"[auth] list_tools by {user}")
        return self._inner.list_tools()

    def call_tool(self, name: str, arguments: dict, headers: dict | None = None):
        user = self._authorize(headers)
        print(f"[auth] call_tool {name} by {user}")
        return self._inner.call_tool(name, arguments)


auth_server = AuthenticatedMCPServer(server)

## Happy path — valid token

In [ ]:
good_headers = {"Authorization": "Bearer tok_alice_readwrite"}

print(auth_server.list_tools(headers=good_headers))
contact = auth_server.call_tool("create_contact",
                                {"name": "Jane", "email": "jane@example.com"},
                                headers=good_headers)
contact

## Bad path — invalid token

In [ ]:
bad_headers = {"Authorization": "Bearer tok_someone_else"}

try:
    auth_server.call_tool("create_contact",
                          {"name": "Mallory", "email": "m@evil.test"},
                          headers=bad_headers)
except UnauthorizedError as e:
    print(f"rejected: {e}")

## Worst path — no token at all

In [ ]:
try:
    auth_server.call_tool("create_contact",
                          {"name": "Anon", "email": "anon@nowhere.test"},
                          headers={})
except UnauthorizedError as e:
    print(f"rejected: {e}")

## Mini test

In [ ]:
# Confirm Mallory and Anon could not write
assert "Mallory" not in {c["name"] for c in CONTACTS.values()}, "unauthorized write got through!"
assert "Anon" not in {c["name"] for c in CONTACTS.values()}
# Confirm Jane succeeded
assert any(c["name"] == "Jane" for c in CONTACTS.values())
print("ok")

## The real thing — FastMCP with auth middleware

FastMCP's HTTP transport is built on Starlette/Uvicorn, so you can add ASGI middleware that runs before any tool. This sketch shows the typical shape — production code would use OAuth, JWT verification, or your platform's identity provider.

In [ ]:
REAL_AUTH_SERVER = r'''# sales_mcp_server_auth.py
import os
from mcp.server.fastmcp import FastMCP
from starlette.middleware.base import BaseHTTPMiddleware
from starlette.responses import JSONResponse

API_TOKENS = set(os.environ.get("MCP_API_TOKENS", "").split(","))


class BearerAuthMiddleware(BaseHTTPMiddleware):
    async def dispatch(self, request, call_next):
        # Allow the discovery handshake without a token if you want anonymous
        # discovery; otherwise gate everything.
        auth = request.headers.get("authorization", "")
        if not auth.startswith("Bearer ") or auth[7:] not in API_TOKENS:
            return JSONResponse({"error": "unauthorized"}, status_code=401)
        return await call_next(request)


mcp = FastMCP("sales-tools")

# ...register your @mcp.tool() functions here...

if __name__ == "__main__":
    app = mcp.streamable_http_app()
    app.add_middleware(BearerAuthMiddleware)
    import uvicorn
    uvicorn.run(app, host="0.0.0.0", port=8000)
'''
print(REAL_AUTH_SERVER)

## Preview — scopes and read/write separation

A single boolean "is this token valid?" is the minimum. In real systems you also want:

- **Scopes**: `sales:read` vs `sales:write`. Read-only dashboards should not get tokens that can `create_contact`.
- **Per-tool ACLs**: maybe `delete_contact` requires an additional `admin` scope.
- **Audit trail**: which token called which tool, with what arguments, at what time.

Notebook 05 implements read-vs-write tool separation; notebook 06 layers human approval on top for destructive tools.

## Key takeaway

Auth lives **outside** your tool code, not inside each function. A thin wrapper or middleware is the right place to decide "should this caller be here at all?". Once that's in place, you can keep your tools focused on business logic.